# Figure: Shorkie LM self-attention map over a gene locus

This notebook visualizes the relative-position self-attention learned by the **Shorkie language model (Shorkie LM)** across a 16,384-bp window centered on a yeast gene. For a chosen transformer block (and averaging over heads), it renders the token-by-token attention matrix (`attended-on` vs `attended-by`, both in 128-bp bins) as a heatmap, with the gene/exon/UTR annotations overlaid. The map reveals that attention concentrates within gene bodies and around regulatory boundaries (TSS / UTRs).

**Reproduces:** the per-layer attention-map panel of the Shorkie LM attention-map figure.

**Upstream:** Runs end-to-end from released data — the Shorkie LM weights and the R64 genome. No precomputed attention tensor is needed (an optional cache is documented under `results.attention`).

**Requires:** the `yeast_ml` conda environment with `pip install -e .` (provides `shorkie`, plus `baskerville`, `tensorflow`, `pysam`, `pyranges`). A GPU is optional — the source script forces CPU (`CUDA_VISIBLE_DEVICES=-1`); a single forward pass over 16 kb is feasible on CPU.

**Source script:** `scripts/04_analysis/shorkie_lm/attention_map/1_visualize_attention.py` (LM mode, `--LM_exp`, `attention_offset=74`).

In [ ]:
import gc
import json
import os

# Mirror the source script: extract attention on CPU to keep memory predictable.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '-1')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pysam
import pyranges as pr
import tensorflow as tf
from baskerville import seqnn, layers
from baskerville import gene as bgene

tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

# Real shorkie helpers used below (sequence one-hot + annotation extraction).
from shorkie import config
from shorkie.helpers.yeast_helpers import process_sequence  # noqa: F401  (used via wrapper)

In [ ]:
# --- Path resolution via shorkie.config (never hardcode absolute paths) ---
lm_dir       = config.path('models.shorkie_lm')
params_file  = str(lm_dir / 'train' / 'params.json')
model_file   = str(lm_dir / 'train' / 'model_best.h5')
fasta_file   = str(config.path('genome.fasta'))
gtf_file     = str(config.path('genome.gtf'))

print('LM params :', params_file)
print('LM weights:', model_file)
print('genome    :', fasta_file)
print('gtf       :', gtf_file)

In [ ]:
# --- Load the Shorkie LM (single fold) ---
# The LM input has 4 DNA channels + (165 species + 1) identity channels.
NUM_SPECIES = 165
SEQ_LEN = 16384
ATTENTION_OFFSET = 74   # first transformer block index inside the trunk (LM unet_small_bert_drop)
LAYER_STEP = 11         # stride between successive attention blocks
N_LAYERS = 8            # number of transformer blocks in the LM trunk

with open(params_file) as fh:
    params = json.load(fh)
params_model = params['model']
params_model['num_features'] = NUM_SPECIES + 5   # LM channel count (DNA + species)
params_model['seq_length'] = SEQ_LEN

tf.keras.backend.clear_session()
seqnn_model = seqnn.SeqNN(params_model)
seqnn_model.restore(model_file, trunk=False)
print('Loaded Shorkie LM.')

In [ ]:
# --- Attention extraction (ported verbatim from the source script) ---
# These low-level helpers reimplement the relative-position MHSA softmax using
# baskerville layer internals; they are NOT part of the public shorkie API, so we
# keep them inline here rather than inventing a shorkie symbol.

def _get_attention_weights(self, inputs, training=False):
    seq_len = inputs.shape[1]
    q = self._multihead_output(self._q_layer, inputs)
    k = self._multihead_output(self._k_layer, inputs)
    if self._scaling:
        q *= self._key_size ** -0.5
    content_logits = tf.matmul(q + self._r_w_bias, k, transpose_b=True)
    if self._num_position_features == 0:
        logits = content_logits
    else:
        distances = tf.range(-seq_len + 1, seq_len, dtype=tf.float32)[tf.newaxis]
        positional_encodings = layers.positional_features(
            positions=distances,
            feature_size=self._num_position_features,
            seq_length=seq_len,
            symmetric=self._relative_position_symmetric,
        )
        r_k = self._multihead_output(self._r_k_layer, positional_encodings)
        if self._content_position_bias:
            relative_logits = tf.matmul(q + self._r_r_bias, r_k, transpose_b=True)
        else:
            relative_logits = tf.matmul(self._r_r_bias, r_k, transpose_b=True)
            relative_logits = tf.broadcast_to(
                relative_logits, shape=(1, self._num_heads, seq_len, 2 * seq_len - 1))
        relative_logits = layers.relative_shift(relative_logits)
        logits = content_logits + relative_logits
    return tf.nn.softmax(logits)


def get_attention_model(model, layer_ix, inital_offset=ATTENTION_OFFSET, offset=LAYER_STEP):
    """Keras model that outputs attention weights of transformer block `layer_ix`."""
    trunk = model.model.layers[1]
    return tf.keras.Model(
        trunk.inputs,
        _get_attention_weights(
            trunk.layers[inital_offset + offset * layer_ix],
            trunk.layers[inital_offset + offset * layer_ix - 1].output,
        ),
    )

In [ ]:
# --- Choose a gene and build the model input via shorkie helpers ---
SEARCH_GENE = 'YGL076C'   # one representative gene from the source script's gene_ls

gene_pr = pr.read_gtf(gtf_file)
gene_pr = gene_pr[gene_pr.Feature.isin(['gene', 'exon', 'five_prime_UTR', 'three_prime_UTR'])]
transcriptome = bgene.Transcriptome(gtf_file)
fasta_open = pysam.Fastafile(fasta_file)

gene_key = [k for k in transcriptome.genes if SEARCH_GENE in k][0]
gene = transcriptome.genes[gene_key]
gene_start = gene.get_exons()[0][0]
gene_end = gene.get_exons()[-1][1]
center = (gene_start + gene_end) // 2
start, end = center - SEQ_LEN // 2, center + SEQ_LEN // 2
chrom = gene.chrom
print(f'{SEARCH_GENE}: {chrom}:{start}-{end}')

# shorkie.helpers.yeast_helpers.process_sequence -> (seq one-hot with species channel, annotation_df)
sequence_one_hot, annotation_df = process_sequence(
    fasta_open, chrom, start, end, gene_pr, seq_len=SEQ_LEN, model_type='LM'
)
sequence_one_hot = np.asarray(sequence_one_hot, dtype='float32')
print('input shape:', sequence_one_hot.shape)

In [ ]:
# --- Forward pass: attention per transformer block, averaged fwd + reverse-complement ---
LAYER_INDEX = [5, 6]   # two penultimate blocks, as in the source script default
SCORE_RC = True

layer_maps = []
for layer_ix in LAYER_INDEX:
    att_model = get_attention_model(seqnn_model, layer_ix=layer_ix)
    att = att_model.predict(x=[sequence_one_hot[None, ...]], batch_size=1)  # (1, H, T, T)
    if SCORE_RC:
        att_rc = att_model.predict(x=[sequence_one_hot[None, ::-1, ::-1]], batch_size=1)
        att = 0.5 * (att + att_rc[..., ::-1, ::-1])
    layer_maps.append(att[0])  # (H, T, T)
    del att_model
    gc.collect()

# Average over selected layers and over all heads -> (T, T) attention matrix.
att_matrix = np.mean(np.mean(np.stack(layer_maps, axis=0), axis=0), axis=0)
print('attention matrix shape:', att_matrix.shape)

In [ ]:
# --- Plot the attention heatmap with gene/exon/UTR annotations (ported from plot_attention_score) ---
VMIN, VMAX = 0.001, 0.05
BIN_BP = 128
atten_size = att_matrix.shape[0]

att = att_matrix.copy()
att[att < VMIN] = 0.0

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(att, cmap='hot', vmin=VMIN, vmax=VMAX, aspect='equal')

annotate_features = [
    {'feature': 'gene', 'filter_query': "Strand == '+'", 'plot_type': 'box', 'color': 'lightgreen', 'min_len': 500},
    {'feature': 'gene', 'filter_query': "Strand == '-'", 'plot_type': 'box', 'color': 'deepskyblue', 'min_len': 500},
    {'feature': 'five_prime_UTR', 'filter_query': None, 'plot_type': 'line', 'color': 'magenta', 'min_len': 0},
    {'feature': 'three_prime_UTR', 'filter_query': None, 'plot_type': 'line', 'color': 'magenta', 'min_len': 0},
    {'feature': 'exon', 'filter_query': None, 'plot_type': 'line', 'color': 'deepskyblue', 'min_len': 0},
]

for spec in annotate_features:
    q = "Feature == '" + spec['feature'] + "'"
    if spec['filter_query']:
        q += ' and ' + spec['filter_query']
    fdf = annotation_df.query(q)
    for _, row in fdf.iterrows():
        if (row['End'] - row['Start']) < spec['min_len']:
            continue
        sb = (row['Start'] - start) // BIN_BP
        eb = (row['End'] - start) // BIN_BP
        if spec['plot_type'] == 'box':
            side = min(eb - sb, atten_size - sb)
            ax.add_patch(patches.Rectangle((sb, sb), side, side, linewidth=2,
                                           edgecolor=spec['color'], facecolor='none'))
        else:  # diagonal line marker for exon / UTR
            ax.plot([sb, eb], [sb, eb], linewidth=2, color=spec['color'])

ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_edgecolor('white')
ax.set_xlabel(f'Attended on ({chrom}:{start}-{end})  --->', fontsize=12, loc='left')
ax.set_ylabel(f'Attended by ({chrom}:{start}-{end})  --->', fontsize=12, loc='bottom')
ax.set_title(f'Shorkie LM attention — {SEARCH_GENE} (layers {LAYER_INDEX}, head-avg)', fontsize=12)
plt.tight_layout()
plt.show()